# 06. Baseline Models

## Purpose
Establish performance benchmarks for injury prediction using interpretable,
well-understood models before moving to survival and multi-task approaches in
notebooks 07 and 08.

This notebook:
1. Loads the full-data feature matrix from notebook 05
2. Builds a temporal train/test split that trains on earlier seasons and tests
   on the most recent ones to avoid look-ahead leakage
3. Trains four baselines for the 30-day injury classification task:
   - Naive historical-rate-by-archetype/age-band
   - Logistic Regression with L2 penalty and balanced class weights
   - Random Forest with cross-validated depth and leaf size
   - XGBoost gradient boosting
4. Evaluates all models with AUC-ROC, PR-AUC, Brier score, and calibration
5. Runs walk-forward temporal cross-validation for the strongest model
6. Saves fitted models to `models/` and a comparison table to `reports/tables/`

## Target
Primary target: `injured_next_30d`, a binary label. The same pipeline generalizes
to the 60- and 90-day horizons, which we also report for comparison.

In [ ]:
import sys, json, warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)

PROJECT_ROOT = str(Path('.').resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.baseline_models import (
    prepare_classification_dataset,
    train_logistic_regression,
    train_random_forest,
    train_gradient_boosting,
    train_naive_baseline,
    save_model,
)
from src.models.evaluation import (
    evaluate_classifier,
    plot_calibration_curve,
    compute_feature_importance,
    temporal_cross_validate,
)

MODELS_DIR  = Path('models')
TABLES_DIR  = Path('reports/tables')
FIGURES_DIR = Path('reports/figures')
for d in (MODELS_DIR, TABLES_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('Modules loaded. Output dirs ready.')

## 1. Load Feature Matrix

We use the full feature matrix `feature_matrix.parquet` rather than the
single-season train split. With multiple seasons on disk we can build a genuine
temporal holdout instead of relying on a single-season fallback.

In [ ]:
fm_path = Path('data/processed/feature_matrix.parquet')
if not fm_path.exists():
    raise FileNotFoundError('Run notebook 05 first to build the feature matrix.')

fm = pd.read_parquet(fm_path)
fm['game_date'] = pd.to_datetime(fm['game_date'])

seasons = sorted(fm['season'].unique().tolist())
print(f'Feature matrix: {fm.shape[0]:,} rows x {fm.shape[1]} cols')
print(f'Seasons present: {seasons}')
print(f'Pitchers: {fm["pitcher"].nunique():,}')
print()
print('Label balance:')
for col in ['injured_next_30d', 'injured_next_60d', 'injured_next_90d']:
    rate = fm[col].mean()
    print(f'  {col:20s}  positive rate = {rate:.1%}  (n={int(fm[col].sum())})')

## 2. Temporal Train / Test Split

Models are evaluated on seasons they have never seen, which is the only honest
way to estimate how Injury Risk+ would perform going forward. With the full
2015-2024 pull, the two most recent seasons become the held-out test set and
everything earlier is used for training. The `prepare_classification_dataset`
helper falls back gracefully to a single-season chronological split if only one
season is present, such as when running in TEST_MODE.

In [ ]:
TARGET = 'injured_next_30d'

X_train, X_test, y_train, y_test = prepare_classification_dataset(
    fm, target_col=TARGET,
)

print(f'Train: {X_train.shape[0]:,} rows  | positive rate = {y_train.mean():.1%}')
print(f'Test : {X_test.shape[0]:,} rows  | positive rate = {y_test.mean():.1%}')
print(f'Features: {X_train.shape[1]}')
print(f'Train seasons: {sorted(fm.loc[X_train.index, "season"].unique().tolist())}')
print(f'Test  seasons: {sorted(fm.loc[X_test.index, "season"].unique().tolist())}')

## 3. Train Baseline Models

Four models, increasing in sophistication:

| Model | Why it's here |
|---|---|
| Naive (historical rate × archetype × age band) | Performance floor: anything we ship must beat a lookup table |
| Logistic Regression | Interpretable linear baseline; coefficients are directly inspectable |
| Random Forest | Captures non-linearities and interactions without much tuning |
| XGBoost | Strong tabular baseline; usually the benchmark to beat |

All tree and linear models are wrapped in pipelines that median-impute missing
values before fitting, since rolling-window features are null early in a
pitcher's observed history.

In [ ]:
%%time
models = {}

print('Training naive baseline (historical rate by archetype x age band)...')
models['naive'] = train_naive_baseline(fm.loc[X_train.index], strategy='historical_rate')

print('Training logistic regression...')
models['logistic_regression'] = train_logistic_regression(X_train, y_train)

print('Training random forest (grid search over depth / leaf size)...')
models['random_forest'] = train_random_forest(X_train, y_train)

print('Training XGBoost...')
models['xgboost'] = train_gradient_boosting(X_train, y_train, framework='xgboost')

print('Training LightGBM...')
models['lightgbm'] = train_gradient_boosting(X_train, y_train, framework='lightgbm')

print('\nAll models trained:', list(models.keys()))

## 3b. Hyperparameter Tuning

`FAST_TUNING = True` limits search iterations so this runs on a laptop in
reasonable time. Set `RUN_TUNING = False` to skip and reload saved results.

| Model | Search method | Metric |
|---|---|---|
| Logistic Regression | RandomizedSearchCV | PR AUC |
| Random Forest | RandomizedSearchCV | PR AUC |
| XGBoost | RandomizedSearchCV | PR AUC |

In [ ]:
# Tuning configuration
RUN_TUNING    = True   # Set False to skip and load saved results
FAST_TUNING   = True   # Smaller search space / fewer iterations for laptop
N_ITER_TUNING = 20     # RandomizedSearchCV iterations per model

TUNING_CHECKPOINT = TABLES_DIR / 'hyperparameter_tuning_results.csv'
TUNED_LR_PATH  = MODELS_DIR / 'baseline_logistic_tuned.joblib'
TUNED_RF_PATH  = MODELS_DIR / 'baseline_random_forest_tuned.joblib'
TUNED_XGB_PATH = MODELS_DIR / 'baseline_xgboost_tuned.joblib'
TUNED_LGB_PATH = MODELS_DIR / 'baseline_lightgbm_tuned.joblib'

print(f'RUN_TUNING={RUN_TUNING}  FAST_TUNING={FAST_TUNING}  N_ITER={N_ITER_TUNING}')

if not RUN_TUNING and TUNING_CHECKPOINT.exists():
    tuning_df = pd.read_csv(TUNING_CHECKPOINT)
    print(f'Loaded existing tuning results from {TUNING_CHECKPOINT}')
    display(tuning_df)

In [ ]:
%%time
import joblib as _jl
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform
from xgboost import XGBClassifier

tuning_records = []
models_tuned = {}

if RUN_TUNING:
    n_pos = int(y_train.sum())
    n_neg = int(len(y_train) - n_pos)
    scale_pos = n_neg / n_pos if n_pos > 0 else 1.0

    # Logistic Regression
    print('Tuning Logistic Regression...')
    _lr_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(penalty='l2', max_iter=2000, random_state=42)),
    ])
    _lr_space = {
        'clf__C': loguniform(1e-4, 10.0),
        'clf__class_weight': ['balanced', None],
    }
    _lr_cv = RandomizedSearchCV(
        _lr_pipe, _lr_space, n_iter=N_ITER_TUNING,
        scoring='average_precision', cv=3, random_state=42, n_jobs=-1, refit=True,
    )
    _lr_cv.fit(X_train, y_train)
    models_tuned['logistic_regression'] = _lr_cv.best_estimator_
    tuning_records.append({
        'model': 'logistic_regression', 'best_cv_pr_auc': _lr_cv.best_score_,
        'best_params': str(_lr_cv.best_params_),
    })
    print(f'  Best CV PR AUC = {_lr_cv.best_score_:.4f}  | {_lr_cv.best_params_}')

    # Random Forest
    print('Tuning Random Forest...')
    _n_est = [100, 200] if FAST_TUNING else [200, 400, 600]
    _rf_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(random_state=42, n_jobs=-1)),
    ])
    _rf_space = {
        'clf__n_estimators': _n_est,
        'clf__max_depth': [4, 6, 8, None],
        'clf__min_samples_split': [2, 5, 10],
        'clf__min_samples_leaf': [1, 3, 5],
        'clf__max_features': ['sqrt', 'log2'],
        'clf__class_weight': ['balanced', 'balanced_subsample'],
    }
    _rf_cv = RandomizedSearchCV(
        _rf_pipe, _rf_space, n_iter=N_ITER_TUNING,
        scoring='average_precision', cv=3, random_state=42, n_jobs=-1, refit=True,
    )
    _rf_cv.fit(X_train, y_train)
    models_tuned['random_forest'] = _rf_cv.best_estimator_
    tuning_records.append({
        'model': 'random_forest', 'best_cv_pr_auc': _rf_cv.best_score_,
        'best_params': str(_rf_cv.best_params_),
    })
    print(f'  Best CV PR AUC = {_rf_cv.best_score_:.4f}  | {_rf_cv.best_params_}')

    # XGBoost
    print('Tuning XGBoost...')
    _xgb_n = [100, 200] if FAST_TUNING else [200, 300, 500]
    _xgb_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1)),
    ])
    _xgb_space = {
        'clf__max_depth': [3, 4, 5, 6],
        'clf__learning_rate': [0.01, 0.05, 0.1, 0.15],
        'clf__n_estimators': _xgb_n,
        'clf__subsample': [0.7, 0.8, 0.9],
        'clf__colsample_bytree': [0.7, 0.8, 0.9],
        'clf__min_child_weight': [1, 5, 10],
        'clf__gamma': [0, 0.1, 0.5],
        'clf__reg_alpha': [0, 0.01, 0.1],
        'clf__reg_lambda': [0.5, 1.0, 2.0],
        'clf__scale_pos_weight': [scale_pos],
    }
    _xgb_cv = RandomizedSearchCV(
        _xgb_pipe, _xgb_space, n_iter=N_ITER_TUNING,
        scoring='average_precision', cv=3, random_state=42, n_jobs=-1, refit=True,
    )
    _xgb_cv.fit(X_train, y_train)
    models_tuned['xgboost'] = _xgb_cv.best_estimator_
    tuning_records.append({
        'model': 'xgboost', 'best_cv_pr_auc': _xgb_cv.best_score_,
        'best_params': str(_xgb_cv.best_params_),
    })
    print(f'  Best CV PR AUC = {_xgb_cv.best_score_:.4f}  | {_xgb_cv.best_params_}')

    # LightGBM
    print('Tuning LightGBM...')
    from lightgbm import LGBMClassifier as _LGBMCls
    _lgb_n = [200, 300] if FAST_TUNING else [300, 500, 700]
    _lgb_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', _LGBMCls(random_state=42, n_jobs=-1, verbosity=-1)),
    ])
    _lgb_space = {
        'clf__num_leaves': [31, 63, 127],
        'clf__learning_rate': [0.01, 0.05, 0.1],
        'clf__n_estimators': _lgb_n,
        'clf__subsample': [0.7, 0.8, 0.9],
        'clf__subsample_freq': [1],
        'clf__colsample_bytree': [0.7, 0.8, 0.9],
        'clf__min_child_samples': [10, 20, 50],
        'clf__reg_alpha': [0, 0.01, 0.1],
        'clf__reg_lambda': [0.5, 1.0, 2.0],
        'clf__is_unbalance': [True],
    }
    _lgb_cv = RandomizedSearchCV(
        _lgb_pipe, _lgb_space, n_iter=N_ITER_TUNING,
        scoring='average_precision', cv=3, random_state=42, n_jobs=-1, refit=True,
    )
    _lgb_cv.fit(X_train, y_train)
    models_tuned['lightgbm'] = _lgb_cv.best_estimator_
    tuning_records.append({
        'model': 'lightgbm', 'best_cv_pr_auc': _lgb_cv.best_score_,
        'best_params': str(_lgb_cv.best_params_),
    })
    print(f'  Best CV PR AUC = {_lgb_cv.best_score_:.4f}  | {_lgb_cv.best_params_}')

    tuning_df = pd.DataFrame(tuning_records)
    display(tuning_df[['model', 'best_cv_pr_auc', 'best_params']])

else:
    # Load pre-saved tuned models
    for name, path in [('logistic_regression', TUNED_LR_PATH),
                        ('random_forest', TUNED_RF_PATH),
                        ('xgboost', TUNED_XGB_PATH),
                        ('lightgbm', TUNED_LGB_PATH)]:
        if path.exists():
            models_tuned[name] = _jl.load(path)
            print(f'Loaded {path}')

    tuning_df = pd.read_csv(TUNING_CHECKPOINT) if TUNING_CHECKPOINT.exists() else pd.DataFrame()

print(f'\nTuned models ready: {list(models_tuned.keys())}')

## 4. Evaluate on the Held-Out Test Set

Score every trained baseline on the reserved test seasons and rank them by PR-AUC.

In [ ]:
results = []
test_probs = {}
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    test_probs[name] = proba
    metrics = evaluate_classifier(y_test, proba)
    metrics['model'] = name
    results.append(metrics)

results_df = pd.DataFrame(results).set_index('model')[
    ['auc_roc', 'pr_auc', 'brier_score', 'accuracy', 'precision', 'recall', 'f1', 'mcc']
].sort_values('pr_auc', ascending=False)

print(f'Held-out test set — target = {TARGET}')
display(results_df.style.format('{:.3f}'))

## 4b. Tuned + Calibrated Model Comparison

Compare the tuned models from section 3b against the untuned baselines on the same held-out test set.

**Why tuned models underperformed:** Despite tuning with `scoring='average_precision'` (PR-AUC) in CV, all tuned+calibrated models show lower holdout PR-AUC than the untuned baselines, roughly 0.006 to 0.007 lower across models. The culprit is **isotonic calibration on the 2022 validation set**: isotonic regression fits a non-parametric step function from raw scores to empirical probabilities. With a ~10% positive rate, the fit is dominated by the negative class, mapping most raw scores to near-zero predicted probability and collapsing recall to near zero for XGBoost and LightGBM entirely. This improves Brier score (predicting ~0 for everyone is well-calibrated on an imbalanced dataset) but destroys PR-AUC, which depends on the model actually predicting positives.

This is a known failure mode of isotonic calibration on small, imbalanced val sets: the isotonic mapping overfits to the negative-class majority and removes the discriminative spread the tuned model had found.

**Production model decision:** The **untuned Random Forest** (PR-AUC 0.134, recall 0.794) is used as the final model throughout this project. It is saved to `models/baseline_random_forest.joblib`. The tuned comparison below is retained for methodological transparency: it illustrates why calibration method and val-set composition matter when working with imbalanced clinical and sports outcomes data.

In [ ]:
# Calibrate tuned models (isotonic regression on val set)
# The 2022 validation split was never seen during training/tuning.
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

val_path = Path('data/processed/feature_matrix_val.parquet')
if val_path.exists() and models_tuned:
    val_fm  = pd.read_parquet(val_path)
    _fcols  = X_train.columns.tolist()
    X_val   = val_fm[[c for c in _fcols if c in val_fm.columns]]
    y_val   = val_fm[TARGET]

    calibrated_tuned = {}
    for name, model in models_tuned.items():
        cal = CalibratedClassifierCV(FrozenEstimator(model), method='isotonic')
        cal.fit(X_val, y_val)
        calibrated_tuned[name] = cal
        print(f'  Calibrated {name} (isotonic, n_val={len(y_val):,})')
else:
    calibrated_tuned = {}
    print('No val set or no tuned models — calibration skipped.')

# Evaluate tuned (calibrated) models on the held-out test set
tuned_results = []
for name, model in calibrated_tuned.items():
    proba = model.predict_proba(X_test)[:, 1]
    m = evaluate_classifier(y_test, proba)
    m['model'] = f'{name}_tuned'
    tuned_results.append(m)

if tuned_results:
    tuned_df = pd.DataFrame(tuned_results).set_index('model')[
        ['auc_roc', 'pr_auc', 'brier_score', 'recall', 'f1']
    ].sort_values('pr_auc', ascending=False)
    print('\nTuned + calibrated model test performance:')
    display(tuned_df.style.format('{:.3f}'))

    if len(results_df) > 0:
        print('\nPR AUC improvement (tuned vs. untuned):')
        for name in ['logistic_regression', 'random_forest', 'xgboost']:
            t_name = f'{name}_tuned'
            if name in results_df.index and t_name in tuned_df.index:
                delta = tuned_df.loc[t_name, 'pr_auc'] - results_df.loc[name, 'pr_auc']
                print(f'  {name:25s}: {delta:+.3f}')

# Calibration plots
if calibrated_tuned:
    best_tuned_name = list(calibrated_tuned.keys())[0]
    fig, axes = plt.subplots(1, min(len(calibrated_tuned), 3), figsize=(5 * min(len(calibrated_tuned), 3), 5))
    if len(calibrated_tuned) == 1:
        axes = [axes]
    for ax, (name, model) in zip(axes, list(calibrated_tuned.items())[:3]):
        proba = model.predict_proba(X_test)[:, 1]
        sub_fig = plot_calibration_curve(y_test, proba)
        for line in sub_fig.axes[0].get_lines():
            ax.plot(line.get_xdata(), line.get_ydata(), marker=line.get_marker(), label=line.get_label())
        ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect')
        ax.set_title(f'Calibration — {name} (tuned)')
        ax.set_xlabel('Mean predicted probability')
        ax.set_ylabel('Observed injury rate')
        ax.legend(fontsize=8)
        plt.close(sub_fig)
    fig.tight_layout()
    calib_fig_path = FIGURES_DIR / 'fig_20b_tuned_calibration.png'
    fig.savefig(calib_fig_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved {calib_fig_path}')

## 5. Calibration

A well-calibrated model's predicted probabilities should match observed injury
rates, which matters enormously for Injury Risk+ since the score is built
directly on top of `injury_prob_30d`. We plot reliability diagrams for the two
strongest models.

In [ ]:
best_two = results_df.index[:2].tolist()

fig, axes = plt.subplots(1, len(best_two), figsize=(6 * len(best_two), 5.5))
if len(best_two) == 1:
    axes = [axes]

for ax, name in zip(axes, best_two):
    sub_fig = plot_calibration_curve(y_test, test_probs[name])
    sub_ax = sub_fig.axes[0]
    for line in sub_ax.get_lines():
        ax.plot(line.get_xdata(), line.get_ydata(), marker=line.get_marker(), label=line.get_label())
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Observed injury rate')
    ax.set_title(f'Calibration — {name}')
    ax.legend()
    plt.close(sub_fig)

fig.tight_layout()
fig_path = FIGURES_DIR / 'fig_20_baseline_calibration.png'
fig.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')

## 6. Feature Importance

What is the strongest tree-based model actually using? This is a first look;
notebook 10 covers SHAP-based interpretability in depth.

In [ ]:
top_tree_model_name = results_df.index[results_df.index.isin(['random_forest', 'xgboost'])][0]
top_tree_model = models[top_tree_model_name]

importance_df = compute_feature_importance(top_tree_model, X_train.columns.tolist())
print(f'Top 15 features — {top_tree_model_name}:')
display(importance_df.head(15))

fig, ax = plt.subplots(figsize=(8, 6))
top15 = importance_df.head(15).iloc[::-1]
ax.barh(top15['feature'], top15['importance'], color='#4C72B0')
ax.set_xlabel('Importance')
ax.set_title(f'Top 15 features — {top_tree_model_name}')
fig.tight_layout()
fig_path = FIGURES_DIR / 'fig_21_baseline_feature_importance.png'
fig.savefig(fig_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')

## 7. Walk-Forward Temporal Cross-Validation

A single train/test split can be lucky or unlucky. Walk-forward CV trains on
seasons `[..., k]` and tests on season `k+1`, repeating across the available
history, which gives a much more honest read on stability than one holdout. We
run it for the strongest model from the comparison above.

In [ ]:
best_model_name = results_df.index[0]
print(f'Running temporal CV for: {best_model_name}')

if best_model_name == 'logistic_regression':
    model_fn = train_logistic_regression
elif best_model_name == 'random_forest':
    model_fn = train_random_forest
elif best_model_name == 'xgboost':
    model_fn = lambda X, y: train_gradient_boosting(X, y, framework='xgboost')
elif best_model_name == 'lightgbm':
    model_fn = lambda X, y: train_gradient_boosting(X, y, framework='lightgbm')
else:
    model_fn = train_logistic_regression

cv_df = temporal_cross_validate(model_fn, fm, n_splits=5, target_col=TARGET)
display(cv_df[['fold', 'test_season', 'n_test', 'auc_roc', 'pr_auc', 'brier_score']])

if len(cv_df):
    print()
    print(f'Mean AUC-ROC across folds: {cv_df["auc_roc"].mean():.3f} '
          f'(std {cv_df["auc_roc"].std():.3f})')

## 8. Multi-Horizon Comparison: 30, 60, and 90 Days

How does the strongest model's discrimination change as the prediction horizon
widens? Longer horizons are easier to predict because they have more positive
cases and need less precise timing, but they are less actionable.

In [ ]:
horizon_rows = []
for horizon_col in ['injured_next_30d', 'injured_next_60d', 'injured_next_90d']:
    Xh_train, Xh_test, yh_train, yh_test = prepare_classification_dataset(fm, target_col=horizon_col)
    if best_model_name == 'logistic_regression':
        m = train_logistic_regression(Xh_train, yh_train)
    elif best_model_name == 'random_forest':
        m = train_random_forest(Xh_train, yh_train)
    elif best_model_name == 'xgboost':
        m = train_gradient_boosting(Xh_train, yh_train, framework='xgboost')
    elif best_model_name == 'lightgbm':
        m = train_gradient_boosting(Xh_train, yh_train, framework='lightgbm')
    else:
        m = train_logistic_regression(Xh_train, yh_train)
    proba = m.predict_proba(Xh_test)[:, 1]
    metrics = evaluate_classifier(yh_test, proba)
    metrics['horizon'] = horizon_col
    metrics['positive_rate_test'] = float(yh_test.mean())
    horizon_rows.append(metrics)

horizon_df = pd.DataFrame(horizon_rows).set_index('horizon')[
    ['positive_rate_test', 'auc_roc', 'pr_auc', 'brier_score', 'f1']
]
display(horizon_df.style.format('{:.3f}'))

## 9. Save Models and Results

Persist the fitted baselines, tuned variants, calibrated models, and metric
tables to `models/` and `reports/tables/` for downstream notebooks.

In [ ]:
# Save untuned baseline models
_model_name_map = {
    'logistic_regression': 'baseline_logistic',
    'random_forest':       'baseline_random_forest',
    'xgboost':             'baseline_xgboost',
    'naive':               'baseline_naive',
    'lightgbm':            'baseline_lightgbm',
}
for name, model in models.items():
    fname = _model_name_map.get(name, f'baseline_{name}')
    out_path = MODELS_DIR / f'{fname}.joblib'
    save_model(model, str(out_path))
    print(f'Saved {out_path}')

# Save tuned (uncalibrated) and calibrated tuned models
for name, model in models_tuned.items():
    fname = _model_name_map.get(name, f'baseline_{name}')
    out_path = MODELS_DIR / f'{fname}_tuned.joblib'
    save_model(model, str(out_path))
    print(f'Saved {out_path}')

for name, model in calibrated_tuned.items():
    fname = _model_name_map.get(name, f'baseline_{name}')
    out_path = MODELS_DIR / f'{fname}_tuned_calibrated.joblib'
    save_model(model, str(out_path))
    print(f'Saved {out_path}')

# Save metrics tables
results_out = results_df.reset_index()
results_out.to_csv(TABLES_DIR / 'baseline_model_metrics.csv', index=False)
print(f'\nSaved {TABLES_DIR / "baseline_model_metrics.csv"}')

if tuning_records:
    pd.DataFrame(tuning_records).to_csv(TABLES_DIR / 'hyperparameter_tuning_results.csv', index=False)
    print(f'Saved {TABLES_DIR / "hyperparameter_tuning_results.csv"}')

if tuned_results:
    tuned_df.reset_index().to_csv(TABLES_DIR / 'tuned_baseline_model_metrics.csv', index=False)
    print(f'Saved {TABLES_DIR / "tuned_baseline_model_metrics.csv"}')

if 'cv_df' in dir() and len(cv_df):
    cv_df.to_csv(TABLES_DIR / 'baseline_temporal_cv_results.csv', index=False)
    print(f'Saved {TABLES_DIR / "baseline_temporal_cv_results.csv"}')

if 'horizon_df' in dir():
    horizon_df.reset_index().to_csv(TABLES_DIR / 'baseline_horizon_comparison.csv', index=False)
    print(f'Saved {TABLES_DIR / "baseline_horizon_comparison.csv"}')

## 10. Summary

* **Best model:** see the AUC-ROC ranking in section 4. This becomes the
  benchmark that survival and multi-task models in notebooks 07 and 08 must beat.
* **Calibration:** reliability diagrams in section 5 show whether predicted
  probabilities can be trusted at face value, or whether `score_calibration`
  in notebook 09 needs to apply isotonic or Platt correction before they feed
  Injury Risk+.
* **Stability:** walk-forward CV in section 7 is a more trustworthy estimate
  of real-world performance than any single train/test split.
* **Next step:** notebook 07 reframes this as a time-to-event problem, where
  survival models can use *all* pitchers including censored ones rather than
  just those with a binary label at a fixed horizon.

In [ ]:
provenance = {
    'notebook': '06_baseline_models',
    'run_at': datetime.now(timezone.utc).isoformat(),
    'seasons_used': seasons,
    'n_train': int(len(X_train)),
    'n_test': int(len(X_test)),
    'target': TARGET,
    'best_model': best_model_name,
    'test_metrics': results_df.loc[best_model_name].to_dict(),
}
print(json.dumps(provenance, indent=2, default=str))

prov_path = TABLES_DIR / 'baseline_model_provenance.json'
prov_path.write_text(json.dumps(provenance, indent=2, default=str))
print(f'\nSaved {prov_path}')